# Validation A — calculated emittance

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/radcoolpv-py/blob/main/docs/site/notebooks/validation_a_optics.ipynb)

**Akerboom *et al.*, *ACS Photonics* 9 (2022) 3831–3840,
[doi:10.1021/acsphotonics.2c01389](https://doi.org/10.1021/acsphotonics.2c01389)**

## The physics

A silicon module radiates heat to the sky through the 8–13 µm atmospheric
window. Bare silicon under a gold back reflector barely emits there at all, so
it runs hot. Silica does emit strongly in that band — the Si–O stretching
resonance — so a silica layer turns the module into a thermal emitter.

Patterning that silica into microcylinders adds a second effect. The array is
comparable in size to the wavelength, so it behaves as a graded index between
air and silica, suppressing reflection and letting the surface emit closer to a
blackbody across a wider band.

Emittance is computed from the geometry with RCWA. The gold blocks
transmission, so $\epsilon = 1 - R$, and Kirchhoff's law lets the absorptance be
read as emittance. Silicon is modeled as **nonabsorbing**, the paper's own
stated assumption for the cooling band.

## Main result

Mean emittance over 7.5–16 µm, against the paper's Figure 3a:

| Surface | radcoolpv | Digitized | Paper text |
|---|---:|---:|---:|
| Bare Au/Si | 0.032 | 0.036 | ~3.5% |
| Flat silica | 0.842 | 0.843 | — |
| Silica cylinders | 0.984 | 0.977 | — |

The cylinders emit almost as a blackbody; the bare module is nearly transparent
to its own heat. The cell below reproduces that table.

## Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline, report
print("radcoolpv ready in", PROJECT)

## The case

This is the YAML that produced the cylinder spectrum above — the paper's
structure, written out in full. Edit anything: the cylinder radius and height,
the pitch, the layer thicknesses, the wavelength range.

`lattice.x` is `6.125 × √3`, because radcoolpv builds a hexagonal lattice as a
centered-rectangular cell with a second cylinder at `(x/2, y/2)` — that is only
hexagonal when `x = y√3`. Setting `photonic_material: vacuum` with
`shape: flat` and dropping the silica layer gives the bare reference.

In [ ]:
%%writefile validation_a.yaml
run:
  optics: true
  thermal: false
  plots: true
  mode: standard
  write_outputs: true
  results_dir: results/validation_a

simulation:
  wavelength: {min: 2.0, max: 16.0, n: 281}
  angles: normal
  polarization: unpolarized
  s4_modes: 60

geometry:
  source: s4
  shape: cylinder               # flat -> the unpatterned silica reference
  photonic_material: sio2       # vacuum -> the bare Au/Si reference
  lattice: {type: hexagonal, x: 10.608811, y: 6.125}
  cylinder: {radius: 1.75, height: 2.25}

structure:
  - {material: sio2, thickness: 500.0}
  - {material: silicon, thickness: 500.0}
  - {material: gold, thickness: 0.08}
  - {material: vacuum, thickness: 0.0, terminal: true}

materials:
  sio2: PalikKitamura_SiO2
  silicon: Akerboom_Si_lossless
  gold: RII_Olmon_2012_ev_Au

comparison:
  spectra:
    - {label: Paper Fig. 3a, file: validation/data/fig3a_calculated_emittance.txt,
       column: 3, color: '#0000ff'}

In [ ]:
# The two switches at the end of this notebook act on this case.
CASE = "validation_a.yaml"

## The result

These three spectra were computed from the geometry below with S4 at its
converged settings and committed, so this notebook reproduces the published
table in seconds without a solver. The last section recomputes them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from radcoolpv.optics.averages import band_average

SPECTRA = {"bare Au/Si":       "validation/data/A1_optics_bare.txt",
           "flat silica":      "validation/data/A2_optics_flat_silica.txt",
           "silica cylinders": "validation/data/A3_optics_cylinders.txt"}
PAPER = np.loadtxt("validation/data/fig3a_calculated_emittance.txt")

fig, ax = plt.subplots(figsize=(7, 4))
rows = []
for column, (label, path) in enumerate(SPECTRA.items(), start=1):
    lam, emit = np.loadtxt(path, usecols=(0, 3), unpack=True)
    line, = ax.plot(lam, emit, label=label)
    ax.plot(PAPER[:, 0], PAPER[:, column], "--", color=line.get_color(), alpha=0.6)
    rows.append((label, band_average(lam, emit, 7.5, 16.0),
                 band_average(PAPER[:, 0], PAPER[:, column], 7.5, 16.0)))

ax.set(xlabel="wavelength (um)", ylabel="emittance", ylim=(0, 1.05))
ax.legend(title="solid: radcoolpv    dashed: paper Fig. 3a")
plt.show()

display(Markdown("| Surface | radcoolpv | Digitized Fig. 3a |\n|---|---:|---:|\n"
                 + "\n".join(f"| {n} | {a:.3f} | {b:.3f} |" for n, a, b in rows)))

## Run it on your own data

Leave `MY_DATA = False` and this cell does nothing — the case above has already
run. Set it to `True` and it opens a file picker, wires your file into the same
case, and runs it. You never have to edit the YAML to use your own spectrum.

Your file needs wavelength in micrometres in the first column and emittance in
another; set `MY_COLUMN` to that column's index. If you are handing it a
spectrum radcoolpv itself exported, set `MY_COLUMN = None` instead — those
files already say which column is which.

**If your spectrum reaches below about 1.1 µm you also get the PV parameters.**
Above the band gap essentially everything absorbed is absorbed in the silicon,
so radcoolpv takes the silicon absorptance to equal the emittance there and
zero below; `run.json` records that this was assumed rather than solved.

In [ ]:
MY_DATA = False      # True -> pick a file and run this case on it
MY_COLUMN = 1         # emittance column; None if the file is a radcoolpv export

if MY_DATA:
    from google.colab import files
    name = next(iter(files.upload()))      # Colab saves it beside the notebook
    cfg = config.load_cases(CASE)[0]
    cfg.run.optics = False
    cfg.run.optics_results = name
    cfg.run.optics_results_emittance_column = MY_COLUMN
    report.summary(pipeline.run(cfg))
else:
    print("Ran the case above. Set MY_DATA = True to run it on your own file.")

## Recompute the optics from the geometry

The run above read a spectrum this repository computed once and committed, so it
reproduces the published numbers in seconds on a machine with no solver. To
compute it yourself from the geometry in the YAML, set the switch below.

S4 is a C++ extension with no PyPI package, so it is built from source: about
ten minutes, then about two minutes of solving. Leave the switch off for a normal run.

A different material goes in here too: upload a CSV whose first line is
`lambda_um,n,k` into `radcoolpv/materials/data/`, then name it (without the
`.csv`) in the `materials:` block above before switching this on.

In [ ]:
RECOMPUTE_WITH_S4 = False       # True -> build S4 and solve the YAML above

S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"      # the tested revision

def build_s4():
    import importlib, importlib.util
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable."); return
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential", "git",
                    "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
                    "libopenblas-dev", "libsuitesparse-dev"], check=True)
    src = Path("/content/S4")
    if not src.exists():
        subprocess.run(["git", "clone", "https://github.com/phoebe-p/S4.git",
                        str(src)], check=True)
    subprocess.run(["git", "checkout", S4_COMMIT], cwd=src, check=True)
    subprocess.run(["make", "-j2", "S4_pyext"], cwd=src, check=True)
    importlib.invalidate_caches()

if RECOMPUTE_WITH_S4:
    build_s4()
    report.summary(pipeline.run(config.load_cases(CASE)[0]))
else:
    print("Using the committed spectrum. Set RECOMPUTE_WITH_S4 = True to solve"
          " the YAML above instead.")

**Exercise.** The three spectra differ by one layer and one pattern. Integrate
each over 8–13 µm instead of 7.5–16 µm with `band_average` and see how much of
the cylinders' advantage sits inside the atmospheric window, where it can
actually reach space, and how much sits outside it.